# LeetCode 1156 — Swap For Longest Repeated Character Substring bằng Python

Link đề: https://leetcode.com/problems/swap-for-longest-repeated-character-substring/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def maxRepOpt1(self, text: str) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên** — độ dài lớn nhất của một đoạn con (substring) gồm **toàn một ký tự**, sau khi được phép **đổi chỗ (swap) tối đa một cặp ký tự**. Mấu chốt: chỉ ký tự `c` nào đó "lấp" được một lỗ hổng nếu ở **nơi khác** còn một ký tự `c` dư; độ dài luôn bị chặn bởi **tổng số lần xuất hiện** của `c`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một lần swap hai vị trí bất kỳ (không cần kề nhau) rồi đo đoạn liên tiếp dài nhất gồm một ký tự.
2. Nắm công cụ: `collections.Counter` (đếm tần suất), `itertools.groupby` (nén chuỗi thành các **nhóm chạy** — run-length encoding), cửa sổ trượt (sliding window).
3. Đi từ **thử mọi cặp swap** → **thử mọi đoạn con** → **cửa sổ trượt theo từng ký tự** → **ghép nhóm chạy O(n)**.
4. Hiểu hai quy tắc chặn: "kéo dài một nhóm thêm 1" và "nối hai nhóm cách nhau đúng 1 ký tự", cả hai bị chặn bởi `count[c]`.
5. Tránh lỗi kinh điển: quên chặn bằng `count[c]`, nối hai nhóm cách nhau ≥ 2 ký tự, quên trường hợp không cần swap.

## 2. Đề bài & ví dụ

Cho chuỗi `text`. Bạn **được phép** đổi chỗ hai ký tự bất kỳ trong chuỗi (tối đa một lần; cũng có thể không đổi).

Trả về độ dài lớn nhất của một **đoạn con liên tiếp** mà mọi ký tự trong đoạn đều giống nhau.

**Ví dụ 1:** `text = "ababa"` → `3`. Đổi `b` đầu tiên với `a` cuối cùng → `"aaabb"`, có `"aaa"` dài 3.

**Ví dụ 2:** `text = "aaabaaa"` → `6`. Đổi `b` với `a` cuối → `"aaaaaab"`, có đoạn `"aaaaaa"` dài 6 (không thể 7 vì chỉ có 6 chữ `a`).

**Ví dụ 3:** `text = "aaaaa"` → `5`. Không cần đổi.

Thêm hai ví dụ hay (từ phiên bản cũ của đề): `"aaabbaaa"` → `4`, `"abcdef"` → `1`.

**Ràng buộc:** `1 <= text.length <= 2 * 10^4`; `text` chỉ gồm chữ thường tiếng Anh.

In [1]:
import random
import time
from collections import Counter
from itertools import groupby

examples = [
    ("ababa", 3),
    ("aaabaaa", 6),
    ("aaaaa", 5),
    ("aaabbaaa", 4),
    ("abcdef", 1),
]
for text, want in examples:
    print(f"{text!r:12} -> {want}")

'ababa'      -> 3
'aaabaaa'    -> 6
'aaaaa'      -> 5
'aaabbaaa'   -> 4
'abcdef'     -> 1


## 3. Kiến thức nền

### 3.1 Đoạn con (substring) và "đoạn lặp một ký tự"

**Substring** là một dãy ký tự **liên tiếp** của chuỗi, ví dụ `text[i:j]`. Một đoạn "lặp một ký tự" là đoạn mà `len(set(đoạn)) == 1`. Không có swap, đáp án chỉ là độ dài **nhóm chạy** (run) dài nhất — dãy ký tự giống nhau liên tiếp tối đa.

In [2]:
def longest_run(s):
    """Độ dài nhóm ký tự giống nhau liên tiếp dài nhất (không swap)."""
    best = cur = 0
    for i, ch in enumerate(s):
        cur = cur + 1 if i > 0 and s[i - 1] == ch else 1
        best = max(best, cur)
    return best

print(longest_run("aaabaaa"), longest_run("ababa"), longest_run("abbbbc"))
assert longest_run("aaabaaa") == 3 and longest_run("ababa") == 1 and longest_run("abbbbc") == 4

3 1 4


### 3.2 `collections.Counter` — đếm tần suất

`Counter(text)` trả về một dict `ký tự → số lần xuất hiện`. Trong bài này `cnt[c]` là **giới hạn trên tuyệt đối**: không cách swap nào tạo được đoạn `c` dài hơn số chữ `c` có trong chuỗi.

In [3]:
cnt = Counter("aaabaaa")
print(cnt, cnt["a"], cnt["z"])  # truy cập khoá không tồn tại trả về 0, không lỗi
assert cnt["a"] == 6 and cnt["b"] == 1 and cnt["z"] == 0

Counter({'a': 6, 'b': 1}) 6 0


### 3.3 `itertools.groupby` — nén chuỗi thành nhóm chạy (run-length encoding)

`groupby(s)` gom các phần tử **liên tiếp** bằng nhau. Kết hợp với `len(list(g))` ta được danh sách `(ký tự, độ dài)`. Đây gọi là **run-length encoding** (mã hoá độ dài loạt).

In [4]:
def runs(s):
    return [(ch, len(list(g))) for ch, g in groupby(s)]

print(runs("aaabaaa"))
print(runs("aabbbaac"))
assert runs("aaabaaa") == [("a", 3), ("b", 1), ("a", 3)]

[('a', 3), ('b', 1), ('a', 3)]
[('a', 2), ('b', 3), ('a', 2), ('c', 1)]


### 3.4 Swap trên chuỗi Python

Chuỗi Python **bất biến** (immutable), nên muốn đổi chỗ hai ký tự ta chuyển sang list, đổi, rồi `"".join`. Cách này chỉ dùng trong lời giải brute force.

In [5]:
def swapped(s, i, j):
    a = list(s)
    a[i], a[j] = a[j], a[i]
    return "".join(a)

print(swapped("ababa", 1, 4))
assert swapped("ababa", 1, 4) == "aaabb"

aaabb


### 3.5 Cửa sổ trượt (sliding window)

Cửa sổ trượt duy trì đoạn `[left, right]`; mỗi bước mở rộng `right`, và khi đoạn vi phạm điều kiện thì tăng `left`. Mỗi chỉ số vào/ra cửa sổ tối đa một lần → tổng `O(n)`. Ví dụ nhỏ: đoạn dài nhất có **tối đa một** ký tự khác `'a'`.

In [6]:
def longest_with_one_other(s, c):
    left = bad = best = 0
    for right, ch in enumerate(s):
        bad += ch != c
        while bad > 1:
            bad -= s[left] != c
            left += 1
        best = max(best, right - left + 1)
    return best

print(longest_with_one_other("aaabaaa", "a"), longest_with_one_other("aabbaa", "a"))
assert longest_with_one_other("aaabaaa", "a") == 7
assert longest_with_one_other("aabbaa", "a") == 3

7 3


## 4. Ý tưởng nền tảng / trực giác

Tính tay `text = "aaabaaa"`:

- Nhóm chạy: `a×3, b×1, a×3`. `cnt[a] = 6`.
- Nhóm `a×3` đầu: nếu đem một `a` từ nơi khác đến thay `b`, đoạn thành `a×4`… nhưng hay hơn: hai nhóm `a×3` **cách nhau đúng một ký tự** `b`. Nếu thay `b` bằng một `a` thì nối được `3 + 1 + 3 = 7` — nhưng chữ `a` đó phải lấy **từ đâu đó**, mà mọi `a` đều nằm trong hai nhóm này. Lấy `a` ở đầu mút → đoạn còn `6`. Vậy đáp án `min(7, cnt[a]) = 6`.

Tính tay `text = "ababa"`:

- Nhóm: `a,b,a,b,a` (mỗi nhóm dài 1). `cnt[a] = 3`, `cnt[b] = 2`.
- Hai `a` ở vị trí 0 và 2 cách nhau một `b` → nối được `1 + 1 + 1 = 3`, có `a` dư ở vị trí 4 → `min(3, 3) = 3`.

Kết luận trực giác: đoạn tốt nhất luôn có dạng **"một nhóm + 1"** hoặc **"nhóm + 1 lỗ + nhóm"**, và luôn bị cắt bởi `cnt[c]`.

In [7]:
for t in ["aaabaaa", "ababa"]:
    print(t, runs(t), dict(Counter(t)))

aaabaaa [('a', 3), ('b', 1), ('a', 3)] {'a': 6, 'b': 1}
ababa [('a', 1), ('b', 1), ('a', 1), ('b', 1), ('a', 1)] {'a': 3, 'b': 2}


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** thử **mọi** cặp `(i, j)` để swap (và trường hợp không swap), mỗi lần đo `longest_run`. Đúng theo định nghĩa nên rất đáng tin để đối chiếu.

Tối ưu nhỏ: chỉ swap khi `text[i] != text[j]` (đổi hai ký tự giống nhau không thay đổi gì).

In [8]:
def max_rep_brute(text):
    best = longest_run(text)
    n = len(text)
    for i in range(n):
        for j in range(i + 1, n):
            if text[i] != text[j]:
                best = max(best, longest_run(swapped(text, i, j)))
    return best

for text, want in examples:
    got = max_rep_brute(text)
    print(f"{text!r:12} got={got} want={want}")
    assert got == want

'ababa'      got=3 want=3
'aaabaaa'    got=6 want=6
'aaaaa'      got=5 want=5
'aaabbaaa'   got=4 want=4
'abcdef'     got=1 want=1


**Độ phức tạp:** `O(n^2)` cặp swap × `O(n)` để đo → `O(n^3)` thời gian, `O(n)` bộ nhớ. Với `n = 2·10^4` là `8·10^12` phép — không thể.

**Nút thắt:** ta thử cả những swap vô nghĩa (ví dụ đổi hai ký tự ở hai đầu chẳng liên quan đến đoạn tốt nhất). Cần biết **swap nào có ích** mà không thử hết.

## 6. Bước đột phá

Cố định ký tự đích `c` và một đoạn kết quả `[l, r]` gồm toàn `c` sau swap. Một swap thay đổi **tối đa hai** vị trí, trong đó chỉ một vị trí có thể nằm trong đoạn mà chuyển từ "không phải `c`" thành `c`. Vậy:

> **Invariant 1:** trước swap, đoạn `[l, r]` chứa **tối đa một** ký tự khác `c`.
>
> **Invariant 2:** nếu đoạn chứa đúng một ký tự khác `c`, phải có **một chữ `c` ở ngoài đoạn** để đổi vào → độ dài đoạn `≤ cnt[c]`.

Quy tắc "nếu… thì…":

- Nếu đoạn toàn `c` → độ dài hợp lệ (không cần swap), hiển nhiên `≤ cnt[c]`.
- Nếu đoạn có đúng một ký tự lạ và `độ dài ≤ cnt[c]` → hợp lệ (luôn còn `c` ở ngoài, vì trong đoạn chỉ có `độ dài − 1 < cnt[c]` chữ `c`).
- Nếu đoạn có ≥ 2 ký tự lạ → không thể.

Gộp lại: **đáp án = max over c của min(cnt[c], đoạn dài nhất có ≤ 1 ký tự khác c)**.

Nhìn theo nhóm chạy, đoạn như vậy chỉ có hai dạng: nhóm `c` dài `L` cộng thêm 1 ô lạ (`L + 1`), hoặc hai nhóm `c` dài `L1`, `L2` kẹp **đúng một** ô lạ (`L1 + 1 + L2`).

In [9]:
# Kiểm chứng nhanh công thức "max_c min(cnt[c], longest_with_one_other)" trên các ví dụ
def formula(text):
    cnt = Counter(text)
    return max(min(cnt[c], longest_with_one_other(text, c)) for c in cnt)

for text, want in examples:
    print(text, formula(text), want)
    assert formula(text) == want

ababa 3 3
aaabaaa 6 6
aaaaa 5 5
aaabbaaa 4 4
abcdef 1 1


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Thử mọi đoạn con (O(n²))

Áp dụng trực tiếp Invariant: với mỗi `l`, mở rộng `r`, đếm số ký tự khác `c` (với `c` thường là `text[l]` hoặc ký tự đa số). Đơn giản hơn: với mỗi ký tự `c` có mặt và mỗi `l`, mở rộng `r` đến khi có 2 ký tự lạ. Đây là `O(26 · n^2)` — vẫn chậm cho `n` lớn nhưng dễ hiểu, tốt để kiểm chứng.

In [10]:
def max_rep_quadratic(text):
    cnt = Counter(text)
    n = len(text)
    best = 0
    for c in cnt:
        for l in range(n):
            bad = 0
            for r in range(l, n):
                bad += text[r] != c
                if bad > 1:
                    break
                best = max(best, min(r - l + 1, cnt[c]))
    return best

for text, want in examples:
    assert max_rep_quadratic(text) == want
print("quadratic OK")

quadratic OK


**Độ phức tạp:** `O(Σ · n^2)` với `Σ ≤ 26` ký tự phân biệt; bộ nhớ `O(Σ)`.

### 7.2 Cách 3 — Cửa sổ trượt cho từng ký tự (O(26·n))

Thay vòng `r` lặp lại từ đầu, dùng cửa sổ trượt `longest_with_one_other(text, c)` đã viết ở mục 3.5: mỗi ký tự `c` một lượt `O(n)`.

In [11]:
def max_rep_window(text):
    cnt = Counter(text)
    best = 0
    for c in cnt:
        left = bad = 0
        for right, ch in enumerate(text):
            bad += ch != c
            while bad > 1:
                bad -= text[left] != c
                left += 1
            best = max(best, min(right - left + 1, cnt[c]))
    return best

for text, want in examples:
    assert max_rep_window(text) == want
print("window OK")

window OK


**Độ phức tạp:** `O(Σ · n)` thời gian (≤ `26 · 2·10^4 ≈ 5·10^5`), `O(Σ)` bộ nhớ. Đã đủ nhanh để nộp.

### 7.3 Cách 4 — Ghép nhóm chạy (O(n), một lượt)

Nén `text` thành danh sách nhóm `groups = [(ch, len), ...]`. Với mỗi nhóm `i`:

1. **Kéo dài thêm 1:** `min(len_i + 1, cnt[ch_i])` — mượn một `ch_i` từ nơi khác đặt cạnh nhóm. Nếu không còn chữ nào ở ngoài (`len_i == cnt`), `min` tự cắt về `len_i`.
2. **Nối qua một lỗ:** nếu nhóm `i+1` dài **đúng 1** và nhóm `i+2` có **cùng ký tự** với nhóm `i`, thì `min(len_i + len_{i+2} + 1, cnt[ch_i])`.

Lấy max qua mọi nhóm.

In [12]:
def max_rep_groups(text):
    cnt = Counter(text)
    groups = [(ch, len(list(g))) for ch, g in groupby(text)]
    best = 0
    for i, (ch, L) in enumerate(groups):
        best = max(best, min(L + 1, cnt[ch]))
        if i + 2 < len(groups) and groups[i + 1][1] == 1 and groups[i + 2][0] == ch:
            best = max(best, min(L + 1 + groups[i + 2][1], cnt[ch]))
    return best

for text, want in examples:
    assert max_rep_groups(text) == want
print("groups OK")

groups OK


**Độ phức tạp:** `O(n)` thời gian (một lượt đếm, một lượt nhóm, một lượt duyệt nhóm), `O(n)` bộ nhớ cho danh sách nhóm (tệ nhất mỗi nhóm dài 1).

### 7.4 Cách 5 — Hai con trỏ không cần mảng nhóm (O(n), O(Σ) bộ nhớ)

Cùng ý tưởng cách 4 nhưng duyệt trực tiếp trên chuỗi: tại `i`, `j` chạy hết nhóm `text[i]` (`[i, j)`), rồi `k` bỏ qua đúng một ký tự và chạy tiếp nhóm cùng ký tự (`[j+1, k)`). Đoạn ghép dài `k - i`; chặn bởi `cnt`. Sau đó nhảy `i = j`.

In [13]:
def max_rep_pointers(text):
    cnt = Counter(text)
    n = len(text)
    best = 0
    i = 0
    while i < n:
        j = i
        while j < n and text[j] == text[i]:
            j += 1
        k = j + 1
        while k < n and text[k] == text[i]:
            k += 1
        # k - i: nhóm [i, j) + 1 ô ở j + nhóm [j+1, k); nếu không có nhóm sau thì là "kéo dài thêm 1"
        best = max(best, min(k - i, cnt[text[i]]))
        i = j
    return best

for text, want in examples:
    assert max_rep_pointers(text) == want
print("pointers OK")

pointers OK


Lưu ý tinh tế: khi `j == n` (nhóm chạm cuối chuỗi), `k = n + 1` nên `k - i = độ dài nhóm + 1` — đúng là trường hợp "kéo dài thêm 1", và `min` với `cnt` sẽ cắt nếu không còn chữ dư. Tuy nhiên cách này chỉ ghép **nhóm với nhóm ở bên phải**; trường hợp kéo dài sang trái vẫn được tính vì nhóm bên trái kia cũng được xét như một `i` riêng và `min(len+1, cnt)` là đối xứng.

## 8. TRACE từng bước

Trace cách 4 (ghép nhóm) trên `text = "aabaaabbba"`:

`cnt = {a: 6, b: 4}`; nhóm: `a2, b1, a3, b3, a1`.

| i | nhóm | kéo dài +1 | nối qua lỗ? | best |
|---|------|-----------|-------------|------|
| 0 | a2 | min(3, 6)=3 | b1 rồi a3 → min(2+1+3, 6)=6 | 6 |
| 1 | b1 | min(2, 4)=2 | a3 dài 3 ≠ 1 → không | 6 |
| 2 | a3 | min(4, 6)=4 | b3 dài 3 → không | 6 |
| 3 | b3 | min(4, 4)=4 | a1 rồi hết → không | 6 |
| 4 | a1 | min(2, 6)=2 | — | 6 |

In [14]:
def trace_groups(text):
    cnt = Counter(text)
    groups = [(ch, len(list(g))) for ch, g in groupby(text)]
    print("cnt =", dict(cnt), "| groups =", groups)
    best = 0
    for i, (ch, L) in enumerate(groups):
        ext = min(L + 1, cnt[ch])
        best = max(best, ext)
        msg = f"i={i} ({ch}{L}) extend={ext}"
        if i + 2 < len(groups) and groups[i + 1][1] == 1 and groups[i + 2][0] == ch:
            join = min(L + 1 + groups[i + 2][1], cnt[ch])
            best = max(best, join)
            msg += f" join_with={groups[i + 2]} -> {join}"
        print(msg, "| best =", best)
    return best

assert trace_groups("aabaaabbba") == 6

cnt = {'a': 6, 'b': 4} | groups = [('a', 2), ('b', 1), ('a', 3), ('b', 3), ('a', 1)]
i=0 (a2) extend=3 join_with=('a', 3) -> 6 | best = 6
i=1 (b1) extend=2 | best = 6
i=2 (a3) extend=4 | best = 6
i=3 (b3) extend=4 | best = 6
i=4 (a1) extend=2 | best = 6


Trace cách 3 (cửa sổ trượt) cho ký tự `c = 'a'` trên cùng chuỗi: theo dõi `left`, `right`, `bad`.

In [15]:
def trace_window(text, c):
    cnt = Counter(text)
    left = bad = best = 0
    for right, ch in enumerate(text):
        bad += ch != c
        while bad > 1:
            bad -= text[left] != c
            left += 1
        cur = min(right - left + 1, cnt[c])
        best = max(best, cur)
        print(f"right={right} ch={ch} left={left} bad={bad} window={text[left:right+1]!r:12} cand={cur} best={best}")
    return best

assert trace_window("aabaaabbba", "a") == 6

right=0 ch=a left=0 bad=0 window='a'          cand=1 best=1
right=1 ch=a left=0 bad=0 window='aa'         cand=2 best=2
right=2 ch=b left=0 bad=1 window='aab'        cand=3 best=3
right=3 ch=a left=0 bad=1 window='aaba'       cand=4 best=4
right=4 ch=a left=0 bad=1 window='aabaa'      cand=5 best=5
right=5 ch=a left=0 bad=1 window='aabaaa'     cand=6 best=6
right=6 ch=b left=3 bad=1 window='aaab'       cand=4 best=6
right=7 ch=b left=7 bad=1 window='b'          cand=1 best=6
right=8 ch=b left=8 bad=1 window='b'          cand=1 best=6
right=9 ch=a left=8 bad=1 window='ba'         cand=2 best=6


## 9. Code nộp LeetCode

Chọn cách 4 (ghép nhóm chạy) — `O(n)`, ngắn gọn, dễ giải thích.

In [16]:
from collections import Counter
from itertools import groupby


class Solution:
    def maxRepOpt1(self, text: str) -> int:
        cnt = Counter(text)
        groups = [(ch, len(list(g))) for ch, g in groupby(text)]
        best = 0
        for i, (ch, length) in enumerate(groups):
            # Mượn một ký tự ch từ nơi khác đặt cạnh nhóm
            best = max(best, min(length + 1, cnt[ch]))
            # Nối hai nhóm cùng ký tự cách nhau đúng 1 ô
            if i + 2 < len(groups) and groups[i + 1][1] == 1 and groups[i + 2][0] == ch:
                best = max(best, min(length + 1 + groups[i + 2][1], cnt[ch]))
        return best


sol = Solution()
for text, want in examples:
    assert sol.maxRepOpt1(text) == want
print("Solution OK")

Solution OK


## 10. Vì sao đúng

**Cận trên.** Xét đáp án tối ưu: sau một swap, đoạn `[l, r]` toàn ký tự `c`. Swap đổi tối đa hai vị trí, nên trước swap đoạn chứa tối đa một ô khác `c`. Đoạn không chứa ô lạ → nó nằm trong một nhóm `c` (độ dài `≤ L`). Đoạn chứa đúng một ô lạ ở vị trí `p` → phần trái của `p` nằm trong nhóm `c` kết thúc tại `p-1`, phần phải nằm trong nhóm `c` bắt đầu tại `p+1`; ô `p` là **một nhóm dài 1** (nếu `p` thuộc nhóm lạ dài ≥ 2 thì một bên của đoạn trống, quay về dạng "nhóm + 1"). Và vì phải đổi một `c` từ ngoài vào, độ dài `≤ cnt[c]`. Như vậy độ dài tối ưu `≤` một trong các ứng viên mà thuật toán xét.

**Đạt được.** Ứng viên `min(L + 1, cnt[c])`: nếu `L < cnt[c]` thì có một `c` ngoài nhóm; đổi nó với ô kề nhóm (ô kề này khác `c` vì nhóm là tối đa; nếu nhóm chạm cả hai đầu chuỗi thì `L = n = cnt[c]`). Ứng viên nối hai nhóm: nếu `L1 + 1 + L2 ≤ cnt[c]` thì còn `c` ở ngoài để đổi vào lỗ; nếu không, lấy `c` ở đầu mút của một nhóm đổi vào lỗ, đoạn còn `L1 + L2 = cnt[c]` khi hai nhóm chứa toàn bộ `c`, nói chung vẫn đạt `min(...)`.

Vậy max của các ứng viên đúng bằng đáp án.

## 11. Time / Space Complexity

Gọi `n = len(text)`, `Σ` = số ký tự phân biệt (`≤ 26`).

| Cách | Time | Space |
|------|------|-------|
| 1. Thử mọi swap | `O(n^3)` | `O(n)` |
| 2. Mọi đoạn con theo từng `c` | `O(Σ · n^2)` | `O(Σ)` |
| 3. Cửa sổ trượt theo từng `c` | `O(Σ · n)` | `O(Σ)` |
| 4. Ghép nhóm chạy | `O(n)` | `O(n)` (danh sách nhóm) |
| 5. Hai con trỏ trên chuỗi | `O(n)` (mỗi `k` có thể quét lại nhóm kế tiếp một lần → vẫn `O(n)` tổng) | `O(Σ)` |

Không có khác biệt average/worst: mọi cách đều duyệt cố định.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [17]:
impls = {
    "brute": max_rep_brute,
    "quadratic": max_rep_quadratic,
    "window": max_rep_window,
    "groups": max_rep_groups,
    "pointers": max_rep_pointers,
    "Solution": Solution().maxRepOpt1,
}
edge = [
    ("a", 1),            # một ký tự
    ("ab", 1),           # swap không giúp
    ("aab", 2),
    ("aba", 2),          # nối qua lỗ nhưng chỉ có 2 chữ a
    ("abab", 2),
    ("aabaa", 4),        # 2+1+2=5 nhưng cnt[a]=4
    ("aabaaa", 5),
    ("aabbaa", 3),       # lỗ dài 2 → không nối được
    ("baaab", 3),        # nhóm chứa toàn bộ chữ a
    ("aaabbbaaa", 4),
    ("abcabc", 2),
]
for name, f in impls.items():
    for text, want in examples + edge:
        assert f(text) == want, (name, text, f(text), want)
print("all impls pass", len(examples + edge), "cases")

all impls pass 16 cases


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

Dùng bảng chữ nhỏ (`"ab"`, `"abc"`) để tạo nhiều nhóm, lỗ, trường hợp biên.

In [18]:
rng = random.Random(1156)
for trial in range(600):
    n = rng.randint(1, 12)
    alphabet = rng.choice(["ab", "abc", "aab", "abcd"])
    text = "".join(rng.choice(alphabet) for _ in range(n))
    want = max_rep_brute(text)
    for name, f in impls.items():
        assert f(text) == want, (name, text)
print("random cross-check OK")

random cross-check OK


### 12.3 Test giới hạn

`n = 2·10^4`. Các cách `O(n)` và `O(26n)` phải chạy rất nhanh.

In [19]:
big_cases = [
    "a" * 20000,
    "ab" * 10000,
    "".join(random.Random(7).choice("abcdefghijklmnopqrstuvwxyz") for _ in range(20000)),
    "a" * 9999 + "b" + "a" * 10000,
]
for text in big_cases:
    t0 = time.perf_counter()
    r1 = Solution().maxRepOpt1(text)
    r2 = max_rep_window(text)
    r3 = max_rep_pointers(text)
    assert r1 == r2 == r3
    print(len(text), r1, f"{time.perf_counter() - t0:.3f}s")

20000 20000 0.006s


20000 3 0.042s
20000 20000 0.008s
20000 19999 0.013s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên chặn bởi `cnt[c]`

Code sai cộng thẳng `L1 + 1 + L2` mà không xét chữ `c` ở lỗ lấy từ đâu.

In [20]:
def wrong_no_cap(text):
    groups = [(ch, len(list(g))) for ch, g in groupby(text)]
    best = 0
    for i, (ch, L) in enumerate(groups):
        best = max(best, L + 1 if len(groups) > 1 else L)
        if i + 2 < len(groups) and groups[i + 1][1] == 1 and groups[i + 2][0] == ch:
            best = max(best, L + 1 + groups[i + 2][1])
    return best

print("sai:", wrong_no_cap("aaabaaa"), "| đúng:", max_rep_groups("aaabaaa"))
assert wrong_no_cap("aaabaaa") == 7 != max_rep_groups("aaabaaa")

sai: 7 | đúng: 6


**Vì sao sai:** chỉ có 6 chữ `a`, không thể có đoạn 7 chữ `a`. **Sửa:** luôn `min(..., cnt[ch])` cho cả hai loại ứng viên (code ở mục 9).

### Lỗi 2 — Nối hai nhóm khi lỗ dài hơn 1

In [21]:
def wrong_any_gap(text):
    cnt = Counter(text)
    groups = [(ch, len(list(g))) for ch, g in groupby(text)]
    best = 0
    for i, (ch, L) in enumerate(groups):
        best = max(best, min(L + 1, cnt[ch]))
        if i + 2 < len(groups) and groups[i + 2][0] == ch:  # quên kiểm tra groups[i+1] dài 1
            best = max(best, min(L + 1 + groups[i + 2][1], cnt[ch]))
    return best

print("sai:", wrong_any_gap("aabbaaa"), "| đúng:", max_rep_groups("aabbaaa"))
assert wrong_any_gap("aabbaaa") != max_rep_groups("aabbaaa")

sai: 5 | đúng: 4


**Vì sao sai:** `"aabbaaa"` có lỗ `bb` dài 2; một swap chỉ lấp được 1 ô → tối đa `4` (`aaa` + 1), không phải `5`. **Sửa:** thêm điều kiện `groups[i + 1][1] == 1`.

### Lỗi 3 — Chỉ trả về nhóm dài nhất + 1 mà không xét có chữ dư hay không

In [22]:
def wrong_plus_one(text):
    return max(L for _, L in runs(text)) + 1

print("sai:", wrong_plus_one("baaab"), "| đúng:", max_rep_groups("baaab"))
print("sai:", wrong_plus_one("aaaaa"), "| đúng:", max_rep_groups("aaaaa"))
assert wrong_plus_one("aaaaa") == 6 and max_rep_groups("aaaaa") == 5

sai: 4 | đúng: 3
sai: 6 | đúng: 5


**Vì sao sai:** `"aaaaa"` không có chữ `a` nào ở ngoài nhóm để mượn; `+1` vượt quá độ dài chuỗi. **Sửa:** `min(L + 1, cnt[ch])`.

### Lỗi 4 — Cửa sổ trượt quên `min(..., cnt[c])`

In [23]:
def wrong_window(text):
    cnt = Counter(text)
    return max(longest_with_one_other(text, c) for c in cnt)

print("sai:", wrong_window("aaabaaa"), "| đúng:", max_rep_window("aaabaaa"))
assert wrong_window("aaabaaa") == 7

sai: 7 | đúng: 6


**Vì sao sai:** cửa sổ `"aaabaaa"` có một ô lạ nhưng không còn `a` nào bên ngoài để đổi vào. **Sửa:** `min(right - left + 1, cnt[c])`.

## 14. Mẹo & nhận dạng pattern

- **"Được thay đổi tối đa k phần tử, tìm đoạn đồng nhất dài nhất"** → cửa sổ trượt "tối đa k phần tử lạ" (họ bài *Longest Repeating Character Replacement* LC 424, *Max Consecutive Ones III* LC 1004).
- **Swap khác replace:** swap không tạo ra ký tự mới, chỉ di chuyển → luôn có cận `cnt[c]`. Đây là điểm khác biệt duy nhất so với LC 424.
- **Chuỗi nhiều nhóm liên tiếp** → nghĩ ngay đến `itertools.groupby` / run-length encoding: bài toán trên chuỗi dài thành bài toán trên danh sách nhóm ngắn hơn.
- **Bảng chữ nhỏ (26)** → được phép lặp qua từng ký tự đích, nhân thêm hệ số 26.
- Khi bài có "một thao tác", hãy hỏi: **thao tác đó thay đổi tối đa bao nhiêu vị trí?** — câu trả lời thường cho ngay invariant.

## 15. Biến thể

1. **Được thay thế (replace) tối đa `k` ký tự** — LC 424: bỏ cận `cnt[c]`, cửa sổ trượt "tối đa k ký tự lạ".
2. **Được swap tối đa `k` lần** — cửa sổ trượt với "tối đa k ký tự lạ" **và** cận `cnt[c]` (mỗi swap đưa một `c` từ ngoài vào). Code: thay `bad > 1` bằng `bad > k`.
3. **Mảng nhị phân, lật tối đa `k` số 0** — LC 1004.
4. **Chỉ được swap hai ký tự kề nhau** — khác hẳn: chỉ có thể dịch chuyển một ký tự đi 1 ô, ứng viên chỉ còn khi nhóm cạnh lỗ.
5. **Xoá đúng một phần tử để có dãy số 1 dài nhất** — LC 1493, cùng kỹ thuật "nối hai nhóm qua một lỗ".

In [24]:
# Biến thể 2: swap tối đa k lần (k = 1 trùng với bài gốc)
def max_rep_k_swaps(text, k):
    cnt = Counter(text)
    best = 0
    for c in cnt:
        left = bad = 0
        for right, ch in enumerate(text):
            bad += ch != c
            while bad > k:
                bad -= text[left] != c
                left += 1
            best = max(best, min(right - left + 1, cnt[c]))
    return best

print(max_rep_k_swaps("aabbaaba", 1), max_rep_k_swaps("aabbaaba", 2))
for text, want in examples:
    assert max_rep_k_swaps(text, 1) == want

4 5


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Brute swap | Thử mọi cặp swap rồi đo | `O(n^3)` | `O(n)` | Kiểm chứng, `n ≤ 50` |
| 2. Mọi đoạn con | Đoạn có ≤ 1 ký tự lạ, chặn `cnt` | `O(Σ n^2)` | `O(Σ)` | Hiểu invariant |
| 3. Cửa sổ trượt | Như cách 2 nhưng hai con trỏ | `O(Σ n)` | `O(Σ)` | Tổng quát hoá được cho k swap |
| 4. Ghép nhóm | Run-length + 2 loại ứng viên | `O(n)` | `O(n)` | Nộp bài, giải thích dễ |
| 5. Hai con trỏ i/j/k | Như cách 4 không cần mảng | `O(n)` | `O(Σ)` | Tiết kiệm bộ nhớ |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `max_rep_groups("abbbabbba")` bằng bao nhiêu? Gợi ý: `cnt[b] = 6`.

**Bài 2 (tính tay):** lập bảng nhóm và ứng viên cho `"aaabbaaab"`.

**Bài 3 (điền TODO):** hoàn thiện hàm đếm nhóm chạy không dùng `groupby`.

**Bài 4 (tự viết lại):** viết lại cách 3 (cửa sổ trượt) nhưng chỉ cho các ký tự xuất hiện ≥ 2 lần (tối ưu nhỏ), vẫn đúng mọi test.

**Bài 5 (mini challenge):** biến thể **chỉ được swap hai ký tự kề nhau**. Viết hàm và đối chiếu brute force.

In [25]:
# Bài 3 — TODO: trả về list (ký tự, độ dài) mà không dùng itertools.groupby
def runs_manual(s):
    # TODO
    raise NotImplementedError

# Test (bỏ comment khi làm xong):
# assert runs_manual("aaabaaa") == [("a", 3), ("b", 1), ("a", 3)]
# assert runs_manual("a") == [("a", 1)]

In [26]:
# Bài 5 — TODO: chỉ được swap hai vị trí kề nhau (i, i+1), tối đa một lần
def max_rep_adjacent(text):
    # TODO
    pass

# Test (bỏ comment khi làm xong):
# assert max_rep_adjacent("aabaa") == 3
# assert max_rep_adjacent("ababa") == 2

## 18. Gợi ý 3 mức

- **Bài 1:** Mức 1: tìm nhóm `b`. Mức 2: `b3, a1, b3` nối qua lỗ được `7`. Mức 3: cắt bởi `cnt[b] = 6`.
- **Bài 2:** Mức 1: nhóm `a3 b2 a3 b1`. Mức 2: lỗ `b2` dài 2 → không nối. Mức 3: ứng viên tốt nhất `min(3 + 1, 6) = 4`.
- **Bài 3:** Mức 1: duyệt và so với ký tự trước. Mức 2: khi khác thì đẩy nhóm cũ vào list. Mức 3: đừng quên đẩy nhóm cuối sau vòng lặp.
- **Bài 4:** Mức 1: ký tự xuất hiện 1 lần cho ứng viên 1. Mức 2: khởi tạo `best = 1`. Mức 3: lọc `c` với `cnt[c] >= 2`.
- **Bài 5:** Mức 1: brute force chỉ có `n - 1` swap. Mức 2: mỗi swap đo `longest_run` → `O(n^2)` đủ cho test nhỏ. Mức 3: phiên bản nhanh chỉ cần xét lỗ dài 1 kẹp giữa nhóm `c` và một `c` đứng ngay sau lỗ… (chính là đổi lỗ với ô kề).

## 19. Đáp án tham khảo

In [27]:
# Bài 1
assert max_rep_groups("abbbabbba") == 6

# Bài 2
assert max_rep_groups("aaabbaaab") == 4
print(runs("aaabbaaab"))

# Bài 3
def runs_manual_sol(s):
    out = []
    for ch in s:
        if out and out[-1][0] == ch:
            out[-1] = (ch, out[-1][1] + 1)
        else:
            out.append((ch, 1))
    return out

assert runs_manual_sol("aaabaaa") == [("a", 3), ("b", 1), ("a", 3)]
assert runs_manual_sol("a") == [("a", 1)]

# Bài 4
def max_rep_window_sol(text):
    cnt = Counter(text)
    best = 1
    for c in cnt:
        if cnt[c] < 2:
            continue
        left = bad = 0
        for right, ch in enumerate(text):
            bad += ch != c
            while bad > 1:
                bad -= text[left] != c
                left += 1
            best = max(best, min(right - left + 1, cnt[c]))
    return best

rng = random.Random(4)
for _ in range(300):
    t = "".join(rng.choice("abc") for _ in range(rng.randint(1, 10)))
    assert max_rep_window_sol(t) == max_rep_brute(t)
print("bài 1-4 OK")

[('a', 3), ('b', 2), ('a', 3), ('b', 1)]
bài 1-4 OK


In [28]:
# Bài 5 — swap kề nhau (brute force O(n^2), đủ cho minh hoạ)
def max_rep_adjacent_sol(text):
    best = longest_run(text)
    for i in range(len(text) - 1):
        if text[i] != text[i + 1]:
            best = max(best, longest_run(swapped(text, i, i + 1)))
    return best

assert max_rep_adjacent_sol("aabaa") == 3
assert max_rep_adjacent_sol("ababa") == 2
assert max_rep_adjacent_sol("aaabaaa") == 4
print("bài 5 OK")

bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
cnt = Counter(text)
groups = [(ch, len(list(g))) for ch, g in groupby(text)]
best = 0
for i, (ch, L) in enumerate(groups):
    best = max(best, min(L + 1, cnt[ch]))                    # kéo dài 1
    if i + 2 < len(groups) and groups[i+1][1] == 1 and groups[i+2][0] == ch:
        best = max(best, min(L + 1 + groups[i+2][1], cnt[ch]))  # nối qua lỗ 1 ô
```

**Checklist:**

- [ ] Mọi ứng viên đều `min(..., cnt[ch])`?
- [ ] Chỉ nối khi lỗ dài **đúng 1** và hai bên **cùng ký tự**?
- [ ] Kiểm tra chỉ số `i + 2 < len(groups)` trước khi truy cập?
- [ ] Chuỗi toàn một ký tự trả về `n` (không phải `n + 1`)?
- [ ] Chuỗi một ký tự (`n = 1`) trả về `1`?
- [ ] Độ phức tạp `O(n)` hoặc `O(26n)` cho `n = 2·10^4`?